# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all record sets available in the dataset and their corresponding fields (columns and `@id`s).

In [ ]:
# Get all record sets using mlcroissant API
for record_set in dataset.record_sets:
    print(f"Record Set @id: {record_set['@id']} | Name: {record_set.get('name','')}")
    print("  Fields:")
    for field in record_set.get('field', []):
        print(f"    Field @id: {field['@id']} | Name: {field.get('name','')} | DataType: {field.get('dataType','')}")
    print("  Columns:")
    for column in record_set.get('column', []):
        print(f"    Column @id: {column['@id']} | Name: {column.get('name','')} | Source: {column.get('source','')}")
    print("\n")

### Example: Print first 5 records for a record set using its `@id`

We'll display records from the primary record set (replace with the actual `@id` if different).

> **Note:** All references should be by `@id`. Below we use an example `@id` representative of the major survey data record set.

In [ ]:
# Find the primary record set (e.g., the main survey responses)

# Typical naming in Croissant for main table:
main_record_set_id = next((r['@id'] for r in dataset.record_sets if 'survey' in r.get('name','').lower()), dataset.record_sets[0]['@id'])
print(f"Using record set @id: {main_record_set_id}")

for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 4:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing only by `@id`.

We'll extract all record sets and place them into DataFrames. All record sets and fields are referenced by their `@id`.

In [ ]:
# Extract data from each record set
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show available columns (all field @id's) for the main record set
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (e.g., GAD-7 score) by its `@id`, filter high scores, normalize, and group by a demographic field.

In [ ]:
# Identify GAD-7 score field @id and a demographic group field @id
df = dataframes[main_record_set_id]
numeric_field_id = next((col for col in df.columns if 'gad7' in col.lower()), None)
group_field_id = next((col for col in df.columns if 'level_of_education' in col.lower()), None)

if numeric_field_id:
    # Ensure numeric field is float/int
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No GAD-7 score field @id found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of GAD-7 scores and show mean scores by level of education.

> All field access is by `@id`, per Croissant schema.

In [ ]:
if numeric_field_id:
    plt.figure(figsize=(8,6))
    df[numeric_field_id].dropna().hist(bins=15)
    plt.title(f"Distribution of GAD-7 Scores ({numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Plot mean GAD-7 by education level (if grouping field exists)
    if group_field_id and group_field_id in df.columns:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean()
        grouped.plot(kind='bar', figsize=(10,6))
        plt.title(f"Mean GAD-7 Score by Education Level ({group_field_id})")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya is AI-ready and rich in demographic and clinical data.
- Survey responses include GAD-7, PHQ-9, and PSQ scores, supporting mental health analysis.
- Data extraction, normalization, filtering, and visualization are directly accessible using `mlcroissant`.
- By referencing entities by their `@id` (fields, record sets), workflows are both reproducible and schema-consistent.